# Company Research Tool

**What this does:** Give it any company URL → it scrapes the website → AI gives you a structured research report.

Useful for: job prep, competitor research, sales calls — anything where you need to quickly understand a company.

---

## How it works (two steps):
1. **Fetch the data** — scrape the company website and strip out the noise
2. **Call the LLM** — send the cleaned text to an AI with a structured prompt, get a research report back

**Models used:** `llama3.2` running locally via Ollama — free, no API key, no internet needed after setup.

## Step 1 — Imports

- `OpenAI` — Python client to talk to any LLM (OpenAI, Ollama, DeepSeek — same client, different URL)
- `requests` — makes HTTP calls to fetch the webpage
- `BeautifulSoup` — parses the raw HTML and lets us extract just the text
- `Markdown / display` — renders the LLM's output as formatted text inside the notebook

In [6]:
from openai import OpenAI
from IPython.display import Markdown, display
import requests
from bs4 import BeautifulSoup

## Step 2 — LLM Client Setup

`OpenAI()` is just an HTTP client. By pointing `base_url` at `localhost:11434`, we talk to Ollama running on this machine instead of OpenAI's cloud.  
`api_key="ollama"` is a dummy value — Ollama doesn't need a real key, but the client requires the field to exist.

To switch to GPT-4: remove `base_url` and `api_key`, load a real key from `.env`, and change `MODEL` to `"gpt-4o"`.

In [7]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "llama3.2"

## Step 3 — ⭐ Scrape the Company Website

Web scraping = fetch the raw HTML of a page → parse it → extract just the readable text.

The challenge: a webpage's HTML is full of noise — navigation menus, footers, scripts, ad slots. If you send all of that to the LLM, it wastes the context window on junk instead of actual company content.

**What this scraper does, line by line:**
1. `requests.get(url)` — fetches the raw HTML of the page (like a browser loading it, but without JavaScript)
2. `BeautifulSoup(...)` — parses that HTML into a tree you can navigate and search
3. `soup.title.string` — pulls out the `<title>` tag (the page name)
4. `tag.decompose()` — removes entire HTML tags and everything inside them. We remove `script`, `style`, `img`, `input` (obvious noise) AND `nav`, `footer`, `header`, `aside` (menus, legal text, sidebars)
5. `get_text(separator="\n")` — extracts all remaining visible text, one line per element
6. `[:5_000]` — truncates to 5,000 characters. LLMs have context limits; 5k is enough for a homepage without overflow

**Try it:** Change `company_url` to any website and run the cell to see the raw scraped text before the LLM touches it.

In [8]:
def fetch_company_data(url):
    response = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })

    if response.status_code != 200:
        return f"Failed to fetch {url} — status {response.status_code}"

    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title"

    # Remove noise: scripts, styles AND structural chrome (nav, footer, etc.)
    for tag in soup(["script", "style", "img", "input", "nav", "footer", "header", "aside"]):
        tag.decompose()

    text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

    return (title + "\n\n" + text)[:5_000]


# YOUR TURN: change this to any company URL
company_url = "https://adobe.com"

website_text = fetch_company_data(company_url)

print(f"Fetched {len(website_text)} characters")
print("---")
print(website_text[:500])  # preview

Fetched 1480 characters
---
Adobe: Creative, marketing and document management solutions

Everything you need to make anything.
Bring any idea to life with products for creators, businesses, and beyond.
See all plans
#FFFFFF
bottom, center, contain
bottom, center, contain
bottom, center, cover
Save {{ccp-intro-offer-percentage}} on Creative Cloud Pro.
Get Photoshop, Illustrator, and more apps plus creative AI. And save for the first 3 months. Terms apply.*
Buy now
#FFFFFF
bottom, center, contain
bottom, center, contain
bot


## Step 3b — Preview: what does the scraped text actually look like?

Run the cell above first, then this cell shows you the raw text before it goes to the LLM.  
This is useful to check: is the content meaningful, or is it mostly nav noise?

In [9]:
print(website_text)  # full scraped text — see exactly what goes into the LLM

Adobe: Creative, marketing and document management solutions

Everything you need to make anything.
Bring any idea to life with products for creators, businesses, and beyond.
See all plans
#FFFFFF
bottom, center, contain
bottom, center, contain
bottom, center, cover
Save {{ccp-intro-offer-percentage}} on Creative Cloud Pro.
Get Photoshop, Illustrator, and more apps plus creative AI. And save for the first 3 months. Terms apply.*
Buy now
#FFFFFF
bottom, center, contain
bottom, center, contain
bottom, center, contain
Adobe Firefly
Save 30% and go unlimited.
Get unlimited generations on select models for a year. Save on Firefly plans until June 17.
Learn more
Background-desktop
Background-tablet
center, top, fill
Background-mobile
center, top, fill
https://main--upp--adobecom.aem.page/homepage/fragments/loggedout/pods/2026/acrobat-183153
https://main--upp--adobecom.aem.page/homepage/fragments/loggedout/pods/superbowl-us-02-05-26/photoshop-176307
https://main--upp--adobecom.aem.page/homepa

## Step 4 — ⭐ The LLM Call

Every LLM API call has the same two parts:

- **`system` prompt** — tells the model what role to play and what format to produce. The model stays in this persona for the whole conversation.
- **`user` prompt** — the actual input. Here we paste the scraped website text and ask for a report.

The output quality depends almost entirely on how well you write the system prompt.  
**Try it:** Change the system prompt — make it write like a VC analyst, a recruiter, or a journalist. The scraped data stays the same, only the lens changes.

In [11]:
system_prompt = """You are a professional business analyst. 
When given a company's website content, produce a clean research report with these sections:
- What they do (1-2 sentences)
- Who they serve (target customers)
- Key products or services
- Company tone / culture signals
- One thing that stands out
Keep it concise. Use bullet points."""

user_prompt = f"Here is the website content for {company_url}:\n\n{website_text}"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

result = response.choices[0].message.content
display(Markdown(result))

**Adobe Research Report**

**What they do:**
* Adobe provides creative, marketing, and document management solutions to help individuals and businesses bring their ideas to life.
* Their products cater to a wide range of users, including creators, businesses, and beyond.

**Who they serve (target customers):**
* Professionals in the creative industry
* Small businesses and entrepreneurs
* Large enterprises and organizations

**Key products or services:**

* Adobe Creative Cloud Pro
* Firefly AI-powered tool for generating visual content
* Acrobat for signing and managing documents
* Marketing and design solutions
* Creative AI technologies

**Company tone / culture signals:**
* Innovation: Emphasis on creating AI-powered tools to "bring any idea to life"
* Creativity: Use of words like "creators," "designers," and "imagination" throughout the website
* Problem-solving: Focus on solving real-world problems for customers, such as managing documents and generating visual content

**One thing that stands out:**
* The use of visually appealing graphics and animations to convey innovation and creativity, making the brand more engaging and memorable.

## Bonus — Person / Profile Analyzer

Same scraper, same LLM call. Only the **system prompt** changes — that's the whole point.

Point it at a personal website, portfolio, or GitHub profile (`github.com/username`).

> **Why not LinkedIn?** LinkedIn actively blocks scrapers and redirects to a login page. You'll get login form HTML, not profile content. Use a public personal site instead.